Imports: standard library, torch/torchvision, and the project's shared modules (`src/*`).

In [1]:

import json
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.datasets import OxfordIIITPet
from torchvision.models import mobilenet_v3_large

from src.dataset import define_transformations
from src.training import device, validate_epoch

Using device: MPS (Apple Silicon GPU)


Load hyperparameters from `configs/base_config.json`.

In [2]:
with open("../configs/base_config.json", encoding="utf-8") as file:
    config_json = json.load(file)

seed = int(config_json["seed"])

batch_size = int(config_json["data"]["batch_size"])
augmentation = config_json["data"]["augmentation"]

model = config_json["model"]["name"]
dropout = config_json["model"]["dropout"]

learning_rate = float(config_json["training"]["learning_rate"])
weight_decay = float(config_json["training"]["weight_decay"])
label_smoothing = float(config_json["training"]["label_smoothing"])
scheduler_config = config_json["training"]["scheduler"]
epochs = int(config_json["training"]["epochs"])

early_stopping_config = config_json["training"]["early_stopping"]
early_stopping_patience = int(early_stopping_config["patience"])
early_stopping_min_delta = float(early_stopping_config["min_delta"])

Pick which experiment's checkpoint to load, from `configs/serving_config.json`.

In [3]:
with open("../configs/serving_config.json", encoding="utf-8") as file:
    serving_config = json.load(file)
EXPERIMENT_DIR = Path(f"../experiments/{serving_config['model_experiment']}")
MODEL_PATH = EXPERIMENT_DIR / "model.pt"
MODEL_CONFIG_PATH = EXPERIMENT_DIR / "config.json"

BATCH_SIZE = 256
NUM_WORKERS = 4

## Load the trained model

In [4]:
with open(MODEL_CONFIG_PATH, encoding="utf-8") as file:
    model_config = json.load(file)

dropout = model_config["model"]["dropout"]

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
train_transform, val_transform = define_transformations(augmentation, IMAGENET_MEAN, IMAGENET_STD)
test_dataset = OxfordIIITPet(
    root="../data",
    split="test",
    target_types="category",
    download=True,
    transform=val_transform
)

loss_function = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model = mobilenet_v3_large(weights=None)
model.classifier[2] = nn.Dropout(p=dropout)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(test_dataset.classes))

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

print(f"Loaded weights from {MODEL_PATH} (dropout={dropout})")

Loaded weights from ../experiments/37/model.pt (dropout=0.5)


Evaluate the loaded model on the true held-out test split.

In [5]:
test_loss, test_accuracy = validate_epoch(model, test_loader, loss_function, device)
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 89.94%
